# Western US 100 Parcel LiDAR-OSM vs Overture Benchmark

This notebook batch-runs the scene-generation comparison from `6_lidar_osm_vs_overture.ipynb` on 100 accepted 500 m x 500 m parcels in the western half of the United States. A parcel is only counted after OSM building footprints cover at least 20% of the parcel area and both building-height modes complete successfully.


In [ ]:
import os
import time
import traceback
from collections import Counter
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
import pandas as pd
import scene_generation.core as core_mod
import shapely
from PIL import Image, ImageDraw
from pyproj import Geod
from shapely.geometry import Polygon
from shapely.ops import unary_union

from scene_generation.core import Scene
from scene_generation.utils import get_utm_epsg_code_from_gps, rect_from_point_and_size


In [ ]:
# Benchmark configuration.
TARGET_COMPLETED_PARCELS = 100
SCENE_WIDTH = 500
SCENE_HEIGHT = 500
MIN_OSM_BUILDING_COVERAGE = 0.20
MIN_CENTER_SEPARATION_M = 550
RANDOM_SEED = 20260612
MAX_CANDIDATE_ATTEMPTS = 5000
CANDIDATE_JITTER_RADIUS_M = 30000

OSM_SERVER = "https://overpass-api.de/api/interpreter"
DATA_ROOT = Path("./scenes/western_us_100_parcels")
RESULTS_DIR = DATA_ROOT / "results"
PARCEL_MANIFEST_CSV = RESULTS_DIR / "completed_parcels.csv"
ATTEMPT_LOG_CSV = RESULTS_DIR / "parcel_attempt_log.csv"

GROUND_MATERIAL_TYPE = "mat-itu_wet_ground"
ROOFTOP_MATERIAL_TYPE = "mat-itu_metal"
WALL_MATERIAL_TYPE = "mat-itu_concrete"

# Keep terrain disabled so the benchmark isolates building-height mode runtime.
ENABLE_LIDAR_TERRAIN = False
ENABLE_DEM_TERRAIN = False

WESTERN_US_BOUNDS = {
    "min_lon": -125.0,
    "max_lon": -100.0,
    "min_lat": 31.0,
    "max_lat": 49.5,
}

# Urban seeds make it feasible to find 500 m x 500 m parcels whose OSM building
# footprint coverage is at least 20%. Candidate parcels are still filtered by
# WESTERN_US_BOUNDS and MIN_OSM_BUILDING_COVERAGE before they count.
WESTERN_US_URBAN_SEEDS = [
    {"seed_name": "Los Angeles CA", "lon": -118.2437, "lat": 34.0522},
    {"seed_name": "San Diego CA", "lon": -117.1611, "lat": 32.7157},
    {"seed_name": "San Francisco CA", "lon": -122.4194, "lat": 37.7749},
    {"seed_name": "Oakland CA", "lon": -122.2711, "lat": 37.8044},
    {"seed_name": "San Jose CA", "lon": -121.8863, "lat": 37.3382},
    {"seed_name": "Sacramento CA", "lon": -121.4944, "lat": 38.5816},
    {"seed_name": "Fresno CA", "lon": -119.7871, "lat": 36.7378},
    {"seed_name": "Long Beach CA", "lon": -118.1937, "lat": 33.7701},
    {"seed_name": "Portland OR", "lon": -122.6784, "lat": 45.5152},
    {"seed_name": "Seattle WA", "lon": -122.3321, "lat": 47.6062},
    {"seed_name": "Tacoma WA", "lon": -122.4443, "lat": 47.2529},
    {"seed_name": "Spokane WA", "lon": -117.4260, "lat": 47.6588},
    {"seed_name": "Phoenix AZ", "lon": -112.0740, "lat": 33.4484},
    {"seed_name": "Tucson AZ", "lon": -110.9747, "lat": 32.2226},
    {"seed_name": "Las Vegas NV", "lon": -115.1398, "lat": 36.1699},
    {"seed_name": "Reno NV", "lon": -119.8138, "lat": 39.5296},
    {"seed_name": "Salt Lake City UT", "lon": -111.8910, "lat": 40.7608},
    {"seed_name": "Denver CO", "lon": -104.9903, "lat": 39.7392},
    {"seed_name": "Colorado Springs CO", "lon": -104.8214, "lat": 38.8339},
    {"seed_name": "Boulder CO", "lon": -105.2705, "lat": 40.0150},
    {"seed_name": "Albuquerque NM", "lon": -106.6504, "lat": 35.0844},
    {"seed_name": "Santa Fe NM", "lon": -105.9378, "lat": 35.6870},
    {"seed_name": "Boise ID", "lon": -116.2023, "lat": 43.6150},
]

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Benchmark output root: {DATA_ROOT.resolve()}")


In [ ]:
geod = Geod(ellps="WGS84")
rng = np.random.default_rng(RANDOM_SEED)


def in_western_us_bounds(lon, lat):
    return (
        WESTERN_US_BOUNDS["min_lon"] <= lon <= WESTERN_US_BOUNDS["max_lon"]
        and WESTERN_US_BOUNDS["min_lat"] <= lat <= WESTERN_US_BOUNDS["max_lat"]
    )


def parcel_points(center_lon, center_lat):
    return rect_from_point_and_size(
        center_lon,
        center_lat,
        "center",
        SCENE_WIDTH,
        SCENE_HEIGHT,
    )


def parcel_polygon_4326(center_lon, center_lat):
    return Polygon(parcel_points(center_lon, center_lat))


def projected_parcel(center_lon, center_lat):
    polygon_4326 = parcel_polygon_4326(center_lon, center_lat)
    utm_crs = get_utm_epsg_code_from_gps(center_lon, center_lat)
    polygon_utm = gpd.GeoSeries([polygon_4326], crs="EPSG:4326").to_crs(utm_crs).iloc[0]
    return polygon_4326, polygon_utm, utm_crs


def random_candidate_center():
    seed = WESTERN_US_URBAN_SEEDS[int(rng.integers(0, len(WESTERN_US_URBAN_SEEDS)))]
    radius_m = float(CANDIDATE_JITTER_RADIUS_M * np.sqrt(rng.random()))
    azimuth = float(rng.uniform(0, 360))
    lon, lat, _ = geod.fwd(seed["lon"], seed["lat"], azimuth, radius_m)
    return {
        "candidate_lon": lon,
        "candidate_lat": lat,
        "seed_name": seed["seed_name"],
        "seed_lon": seed["lon"],
        "seed_lat": seed["lat"],
        "jitter_radius_m": radius_m,
        "jitter_azimuth_deg": azimuth,
    }


def center_distance_m(lon_a, lat_a, lon_b, lat_b):
    _, _, distance_m = geod.inv(lon_a, lat_a, lon_b, lat_b)
    return float(distance_m)


def is_too_close_to_completed(lon, lat, completed_df):
    if completed_df.empty:
        return False
    distances = completed_df.apply(
        lambda row: center_distance_m(lon, lat, row["center_lon"], row["center_lat"]),
        axis=1,
    )
    return bool((distances < MIN_CENTER_SEPARATION_M).any())


In [ ]:
def _polygonal_geometries(gdf):
    if gdf.empty:
        return []
    geometries = []
    for geometry in gdf.geometry:
        if geometry is None or geometry.is_empty:
            continue
        if geometry.geom_type in {"Polygon", "MultiPolygon"}:
            geometries.append(geometry)
    return geometries


def query_osm_building_coverage(center_lon, center_lat):
    """Return OSM building-footprint coverage for one 500 m x 500 m parcel."""
    polygon_4326, polygon_utm, utm_crs = projected_parcel(center_lon, center_lat)
    bbox = polygon_4326.bounds

    try:
        buildings = ox.features.features_from_bbox(bbox=bbox, tags={"building": True})
    except Exception as exc:
        return {
            "coverage_ok": False,
            "osm_building_coverage": 0.0,
            "osm_building_area_m2": 0.0,
            "parcel_area_m2": float(polygon_utm.area),
            "osm_building_count": 0,
            "coverage_error": repr(exc),
        }

    if buildings.empty:
        return {
            "coverage_ok": False,
            "osm_building_coverage": 0.0,
            "osm_building_area_m2": 0.0,
            "parcel_area_m2": float(polygon_utm.area),
            "osm_building_count": 0,
            "coverage_error": "no OSM buildings returned",
        }

    buildings = buildings.to_crs(utm_crs)
    intersecting = buildings[buildings.intersects(polygon_utm)].copy()
    polygonal = _polygonal_geometries(intersecting)

    if not polygonal:
        return {
            "coverage_ok": False,
            "osm_building_coverage": 0.0,
            "osm_building_area_m2": 0.0,
            "parcel_area_m2": float(polygon_utm.area),
            "osm_building_count": 0,
            "coverage_error": "no polygonal OSM buildings intersect parcel",
        }

    clipped = [geometry.intersection(polygon_utm) for geometry in polygonal]
    clipped = [geometry for geometry in clipped if geometry is not None and not geometry.is_empty]
    footprint_union = unary_union(clipped) if clipped else shapely.GeometryCollection()
    building_area_m2 = float(footprint_union.area) if not footprint_union.is_empty else 0.0
    parcel_area_m2 = float(polygon_utm.area)
    coverage = building_area_m2 / parcel_area_m2 if parcel_area_m2 else 0.0

    return {
        "coverage_ok": coverage >= MIN_OSM_BUILDING_COVERAGE,
        "osm_building_coverage": coverage,
        "osm_building_area_m2": building_area_m2,
        "parcel_area_m2": parcel_area_m2,
        "osm_building_count": int(len(polygonal)),
        "coverage_error": "",
    }


In [ ]:
def iter_polygons(geometry):
    """Yield polygon parts from a Shapely Polygon or MultiPolygon."""
    if geometry is None or geometry.is_empty:
        return

    if geometry.geom_type == "Polygon":
        yield geometry
    elif geometry.geom_type == "MultiPolygon":
        yield from geometry.geoms


def rasterize_height_source_mask(records, height_source, shape, ground_bounds):
    """Rasterize footprints for one height source onto a building-map grid."""
    min_x, _, _, max_y = ground_bounds
    mask_image = Image.new("1", (shape[1], shape[0]), 0)
    draw = ImageDraw.Draw(mask_image)

    for record in records:
        if record["height_source"] != height_source:
            continue

        for polygon in iter_polygons(record["footprint"]):
            pixel_coords = [(x - min_x, max_y - y) for x, y in polygon.exterior.coords]
            draw.polygon(pixel_coords, outline=1, fill=1)

    return np.array(mask_image, dtype=bool)


def summarize_height_sources(parcel, mode, height_sources):
    """Summarize counts and numeric percentages by selected height source."""
    total_buildings = int(sum(height_sources.values()))
    rows = []

    for source, building_count in height_sources.most_common():
        building_percentage = (building_count / total_buildings) * 100 if total_buildings else 0.0
        rows.append(
            {
                "parcel_id": parcel["parcel_id"],
                "center_lon": parcel["center_lon"],
                "center_lat": parcel["center_lat"],
                "osm_building_coverage": parcel["osm_building_coverage"],
                "mode": mode,
                "height_source": source,
                "building_count": int(building_count),
                "total_buildings": total_buildings,
                "building_percentage": float(building_percentage),
            }
        )

    return pd.DataFrame(
        rows,
        columns=[
            "parcel_id",
            "center_lon",
            "center_lat",
            "osm_building_coverage",
            "mode",
            "height_source",
            "building_count",
            "total_buildings",
            "building_percentage",
        ],
    )


def compare_height_maps(parcel, lidar_map, overture_map, lidar_records, overture_records, lidar_ground_bounds, overture_ground_bounds):
    """Compute per-parcel height-map comparison statistics."""
    diff = lidar_map.astype(float) - overture_map.astype(float)
    building_mask = (lidar_map > 0) | (overture_map > 0)
    mean_abs_diff = np.mean(np.abs(diff[building_mask])) if building_mask.any() else 0.0
    max_abs_diff = np.max(np.abs(diff)) if diff.size else 0.0

    lidar_hag_mask = rasterize_height_source_mask(
        lidar_records,
        "hag",
        lidar_map.shape,
        lidar_ground_bounds,
    )
    overture_explicit_height_mask = rasterize_height_source_mask(
        overture_records,
        "overture:height",
        lidar_map.shape,
        overture_ground_bounds,
    )
    lidar_hag_not_overture_explicit_height_pixels = int(
        np.count_nonzero(lidar_hag_mask & ~overture_explicit_height_mask)
    )

    return pd.DataFrame(
        [
            {
                "parcel_id": parcel["parcel_id"],
                "center_lon": parcel["center_lon"],
                "center_lat": parcel["center_lat"],
                "osm_building_coverage": parcel["osm_building_coverage"],
                "same_raster": bool(np.array_equal(lidar_map, overture_map)),
                "mean_abs_diff_on_building_pixels_m": float(mean_abs_diff),
                "max_abs_diff_m": float(max_abs_diff),
                "lidar_hag_pixels_outside_overture_explicit_height": lidar_hag_not_overture_explicit_height_pixels,
            }
        ]
    )


In [ ]:
def generate_scene_for_parcel(parcel, mode, out_dir, *, track_height_sources=True):
    """Generate one scene mode and optionally count selected height sources."""
    scene_polygon = parcel_points(parcel["center_lon"], parcel["center_lat"])

    height_sources = []
    height_source_records = []
    original_resolve = core_mod.resolve_building_height

    def logging_resolve_building_height(*args, **kwargs):
        kwargs = dict(kwargs)
        kwargs["return_source"] = True
        height, metadata = original_resolve(*args, **kwargs)
        source = metadata.get("source", "unknown")
        footprint = args[1] if len(args) > 1 else kwargs.get("building_polygon")
        height_sources.append(source)
        height_source_records.append(
            {
                "parcel_id": parcel["parcel_id"],
                "mode": mode,
                "height_source": source,
                "height_m": height,
                "footprint": footprint,
            }
        )
        return height

    if track_height_sources:
        core_mod.resolve_building_height = logging_resolve_building_height

    try:
        scene = Scene()
        building_height_map = scene(
            points=scene_polygon,
            data_dir=str(out_dir),
            osm_server_addr=OSM_SERVER,
            hag_tiff_path=None,
            ground_material_type=GROUND_MATERIAL_TYPE,
            rooftop_material_type=ROOFTOP_MATERIAL_TYPE,
            wall_material_type=WALL_MATERIAL_TYPE,
            generate_building_map=True,
            lidar_terrain=ENABLE_LIDAR_TERRAIN,
            dem_terrain=ENABLE_DEM_TERRAIN,
            building_height_mode=mode,
        )
        ground_bounds = scene._ground_polygon_envelope_UTM.bounds
    finally:
        if track_height_sources:
            core_mod.resolve_building_height = original_resolve

    return building_height_map, Counter(height_sources), height_source_records, ground_bounds


def run_completed_parcel(parcel):
    """Run both modes for one accepted parcel and write per-parcel result CSVs."""
    parcel_dir = DATA_ROOT / parcel["parcel_id"]
    DATA_DIR_LIDAR_OSM = parcel_dir / "lidar_osm"
    DATA_DIR_LIDAR_OVERTURE = parcel_dir / "lidar_overture"
    DATA_DIR_LIDAR_OSM.mkdir(parents=True, exist_ok=True)
    DATA_DIR_LIDAR_OVERTURE.mkdir(parents=True, exist_ok=True)

    runtimes = []

    start_time = time.perf_counter()
    lidar_map, lidar_sources, lidar_records, lidar_ground_bounds = generate_scene_for_parcel(
        parcel,
        "lidar-osm",
        DATA_DIR_LIDAR_OSM,
        track_height_sources=True,
    )
    runtimes.append(
        {
            "parcel_id": parcel["parcel_id"],
            "mode": "lidar-osm",
            "runtime_seconds": time.perf_counter() - start_time,
        }
    )

    start_time = time.perf_counter()
    overture_map, overture_sources, overture_records, overture_ground_bounds = generate_scene_for_parcel(
        parcel,
        "overture",
        DATA_DIR_LIDAR_OVERTURE,
        track_height_sources=True,
    )
    runtimes.append(
        {
            "parcel_id": parcel["parcel_id"],
            "mode": "overture",
            "runtime_seconds": time.perf_counter() - start_time,
        }
    )

    runtime_df = pd.DataFrame(runtimes)
    lidar_runtime = runtime_df.loc[runtime_df["mode"] == "lidar-osm", "runtime_seconds"].iloc[0]
    runtime_df["relative_to_lidar_osm"] = runtime_df["runtime_seconds"] / lidar_runtime if lidar_runtime else np.nan
    runtime_df["center_lon"] = parcel["center_lon"]
    runtime_df["center_lat"] = parcel["center_lat"]
    runtime_df["osm_building_coverage"] = parcel["osm_building_coverage"]

    source_df = pd.concat(
        [
            summarize_height_sources(parcel, "lidar-osm", lidar_sources),
            summarize_height_sources(parcel, "overture", overture_sources),
        ],
        ignore_index=True,
    )
    comparison_df = compare_height_maps(
        parcel,
        lidar_map,
        overture_map,
        lidar_records,
        overture_records,
        lidar_ground_bounds,
        overture_ground_bounds,
    )

    source_df.to_csv(parcel_dir / "height_source_summary.csv", index=False)
    runtime_df.to_csv(parcel_dir / "runtime_summary.csv", index=False)
    comparison_df.to_csv(parcel_dir / "comparison_stats.csv", index=False)

    return source_df, runtime_df, comparison_df


In [ ]:
def load_completed_manifest():
    if PARCEL_MANIFEST_CSV.exists():
        return pd.read_csv(PARCEL_MANIFEST_CSV)
    return pd.DataFrame()


def append_csv_row(path, row):
    path.parent.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame([row])
    df.to_csv(path, mode="a", header=not path.exists(), index=False)


def next_parcel_id(completed_df):
    if completed_df.empty:
        return "parcel_000"
    numeric_ids = completed_df["parcel_id"].str.extract(r"parcel_(\d+)")[0].dropna().astype(int)
    next_id = int(numeric_ids.max()) + 1 if not numeric_ids.empty else len(completed_df)
    return f"parcel_{next_id:03d}"


def load_cached_outputs(completed_df):
    source_frames = []
    runtime_frames = []
    comparison_frames = []

    for parcel_id in completed_df.get("parcel_id", []):
        parcel_dir = DATA_ROOT / parcel_id
        source_path = parcel_dir / "height_source_summary.csv"
        runtime_path = parcel_dir / "runtime_summary.csv"
        comparison_path = parcel_dir / "comparison_stats.csv"
        if source_path.exists() and runtime_path.exists() and comparison_path.exists():
            source_frames.append(pd.read_csv(source_path))
            runtime_frames.append(pd.read_csv(runtime_path))
            comparison_frames.append(pd.read_csv(comparison_path))

    return {
        "height_source_summary": pd.concat(source_frames, ignore_index=True) if source_frames else pd.DataFrame(),
        "runtime_summary": pd.concat(runtime_frames, ignore_index=True) if runtime_frames else pd.DataFrame(),
        "comparison_stats": pd.concat(comparison_frames, ignore_index=True) if comparison_frames else pd.DataFrame(),
    }


def build_parcel_row(parcel_id, candidate, coverage_info):
    return {
        "parcel_id": parcel_id,
        "center_lon": float(candidate["candidate_lon"]),
        "center_lat": float(candidate["candidate_lat"]),
        "data_dir_lidar_osm": str(DATA_ROOT / parcel_id / "lidar_osm"),
        "data_dir_lidar_overture": str(DATA_ROOT / parcel_id / "lidar_overture"),
        "seed_name": candidate["seed_name"],
        "seed_lon": candidate["seed_lon"],
        "seed_lat": candidate["seed_lat"],
        "jitter_radius_m": candidate["jitter_radius_m"],
        "jitter_azimuth_deg": candidate["jitter_azimuth_deg"],
        "osm_building_coverage": coverage_info["osm_building_coverage"],
        "osm_building_area_m2": coverage_info["osm_building_area_m2"],
        "parcel_area_m2": coverage_info["parcel_area_m2"],
        "osm_building_count": coverage_info["osm_building_count"],
        "min_required_osm_building_coverage": MIN_OSM_BUILDING_COVERAGE,
        "scene_width_m": SCENE_WIDTH,
        "scene_height_m": SCENE_HEIGHT,
    }


def delete_existing_tif_files(root_dir):
    """Remove stale TIF/TIFF files before running the full benchmark."""
    deleted = 0
    for path in list(root_dir.rglob("*.tif")) + list(root_dir.rglob("*.tiff")):
        try:
            path.unlink()
            deleted += 1
        except OSError:
            pass
    return deleted


def run_benchmark(target_completed_parcels=TARGET_COMPLETED_PARCELS):
    completed_df = load_completed_manifest()
    completed_count = len(completed_df)
    print(f"Starting with {completed_count} completed parcels in manifest.")

    deleted_tifs = delete_existing_tif_files(DATA_ROOT)
    if deleted_tifs:
        print(f"Deleted {deleted_tifs} existing TIF files under {DATA_ROOT}")

    outputs = load_cached_outputs(completed_df)
    attempts = 0

    while completed_count < target_completed_parcels and attempts < MAX_CANDIDATE_ATTEMPTS:
        attempts += 1
        candidate = random_candidate_center()
        lon = candidate["candidate_lon"]
        lat = candidate["candidate_lat"]

        attempt_row = {
            "attempt": attempts,
            "candidate_lon": lon,
            "candidate_lat": lat,
            "seed_name": candidate["seed_name"],
            "status": "started",
            "message": "",
        }

        if not in_western_us_bounds(lon, lat):
            attempt_row.update({"status": "rejected", "message": "outside western US bounds"})
            append_csv_row(ATTEMPT_LOG_CSV, attempt_row)
            continue

        if is_too_close_to_completed(lon, lat, completed_df):
            attempt_row.update({"status": "rejected", "message": "too close to completed parcel"})
            append_csv_row(ATTEMPT_LOG_CSV, attempt_row)
            continue

        coverage_info = query_osm_building_coverage(lon, lat)
        attempt_row.update(
            {
                "osm_building_coverage": coverage_info["osm_building_coverage"],
                "osm_building_count": coverage_info["osm_building_count"],
            }
        )
        if not coverage_info["coverage_ok"]:
            attempt_row.update({"status": "rejected", "message": coverage_info.get("coverage_error", "coverage below threshold")})
            append_csv_row(ATTEMPT_LOG_CSV, attempt_row)
            continue

        parcel_id = next_parcel_id(completed_df)
        parcel = build_parcel_row(parcel_id, candidate, coverage_info)
        print(
            f"Running {parcel_id}: lon={parcel['center_lon']:.6f}, lat={parcel['center_lat']:.6f}, "
            f"OSM building coverage={parcel['osm_building_coverage']:.3f}"
        )

        try:
            source_df, runtime_df, comparison_df = run_completed_parcel(parcel)
        except Exception as exc:
            attempt_row.update(
                {
                    "status": "scene_generation_failed",
                    "message": repr(exc),
                    "traceback": traceback.format_exc(),
                }
            )
            append_csv_row(ATTEMPT_LOG_CSV, attempt_row)
            print(f"Skipping {parcel_id} after scene-generation failure: {exc!r}")
            continue

        append_csv_row(PARCEL_MANIFEST_CSV, parcel)
        attempt_row.update({"status": "completed", "message": parcel_id})
        append_csv_row(ATTEMPT_LOG_CSV, attempt_row)

        completed_df = load_completed_manifest()
        completed_count = len(completed_df)
        outputs["height_source_summary"] = pd.concat(
            [outputs["height_source_summary"], source_df], ignore_index=True
        )
        outputs["runtime_summary"] = pd.concat(
            [outputs["runtime_summary"], runtime_df], ignore_index=True
        )
        outputs["comparison_stats"] = pd.concat(
            [outputs["comparison_stats"], comparison_df], ignore_index=True
        )
        print(f"Completed {completed_count}/{target_completed_parcels} parcels.")

    if completed_count < target_completed_parcels:
        print(
            f"Stopped after {attempts} attempts with {completed_count}/{target_completed_parcels} completed parcels. "
            "Increase MAX_CANDIDATE_ATTEMPTS or adjust the seed list/jitter radius if needed."
        )

    return {
        "completed_parcels": load_completed_manifest(),
        **load_cached_outputs(load_completed_manifest()),
    }


In [ ]:
# This cell performs the full benchmark. It can take a long time because it runs
# 200 scene-generation jobs after OSM coverage gating. Progress is cached under
# DATA_ROOT, so rerunning the cell resumes from completed parcels.
benchmark_outputs = run_benchmark(TARGET_COMPLETED_PARCELS)

completed_parcels = benchmark_outputs["completed_parcels"]
height_source_summary = benchmark_outputs["height_source_summary"]
runtime_summary = benchmark_outputs["runtime_summary"]
comparison_stats = benchmark_outputs["comparison_stats"]

display(completed_parcels.tail())
print(f"Completed parcels included in aggregations: {len(completed_parcels)}")


In [ ]:
# Height-source aggregation and distributions.
source_counts = height_source_summary.copy()

source_percentage_pivot = (
    source_counts.pivot_table(
        index=["parcel_id", "mode"],
        columns="height_source",
        values="building_percentage",
        aggfunc="sum",
        fill_value=0.0,
    )
    .reset_index()
)

average_height_source_percentages = (
    source_counts.groupby(["mode", "height_source"], as_index=False)["building_percentage"]
    .mean()
    .sort_values(["mode", "building_percentage"], ascending=[True, False])
)

display(average_height_source_percentages)

for mode, mode_df in source_percentage_pivot.groupby("mode"):
    source_columns = [col for col in mode_df.columns if col not in {"parcel_id", "mode"}]
    if not source_columns:
        continue
    axes = mode_df[source_columns].plot(
        kind="hist",
        bins=20,
        alpha=0.55,
        figsize=(12, 5),
        title=f"Distribution of selected height-source percentages: {mode}",
    )
    axes.set_xlabel("Percent of buildings in parcel")
    axes.set_ylabel("Parcel count")
    plt.show()


In [ ]:
# Runtime aggregation and Overture relative-runtime distribution.
runtime_summary_display = runtime_summary.copy()
runtime_summary_display["runtime_seconds"] = runtime_summary_display["runtime_seconds"].round(3)
runtime_summary_display["relative_to_lidar_osm"] = runtime_summary_display["relative_to_lidar_osm"].round(3)
display(runtime_summary_display.head())

average_runtime_by_mode = (
    runtime_summary.groupby("mode", as_index=False)[["runtime_seconds", "relative_to_lidar_osm"]]
    .mean()
    .sort_values("mode")
)
display(average_runtime_by_mode)

overture_relative_runtime = runtime_summary[runtime_summary["mode"] == "overture"].copy()
average_overture_relative_to_lidar_osm = overture_relative_runtime["relative_to_lidar_osm"].mean()
print(f"Average relative_to_lidar_osm for Overture: {average_overture_relative_to_lidar_osm:.3f}")

ax = overture_relative_runtime["relative_to_lidar_osm"].plot(
    kind="hist",
    bins=20,
    figsize=(9, 5),
    title="Distribution of Overture relative_to_lidar_osm runtime",
)
ax.set_xlabel("Overture runtime / LiDAR-OSM runtime")
ax.set_ylabel("Parcel count")
plt.show()


In [ ]:
# Height-map difference aggregation and distributions.
diff_metric_columns = [
    "mean_abs_diff_on_building_pixels_m",
    "max_abs_diff_m",
    "lidar_hag_pixels_outside_overture_explicit_height",
]

average_comparison_stats = comparison_stats[diff_metric_columns].mean().to_frame("average_value")
display(average_comparison_stats)

display(comparison_stats.head())

fig, axes = plt.subplots(1, len(diff_metric_columns), figsize=(18, 4), constrained_layout=True)
for ax, column in zip(axes, diff_metric_columns):
    comparison_stats[column].plot(kind="hist", bins=20, ax=ax)
    ax.set_title(column)
    ax.set_xlabel("Value")
    ax.set_ylabel("Parcel count")
plt.show()


In [ ]:
# Persist aggregate outputs for later reporting.
aggregate_dir = RESULTS_DIR / "aggregates"
aggregate_dir.mkdir(parents=True, exist_ok=True)

average_height_source_percentages.to_csv(aggregate_dir / "average_height_source_percentages.csv", index=False)
average_runtime_by_mode.to_csv(aggregate_dir / "average_runtime_by_mode.csv", index=False)
overture_relative_runtime.to_csv(aggregate_dir / "overture_relative_runtime_by_parcel.csv", index=False)
average_comparison_stats.to_csv(aggregate_dir / "average_comparison_stats.csv")
comparison_stats.to_csv(aggregate_dir / "comparison_stats_by_parcel.csv", index=False)
source_percentage_pivot.to_csv(aggregate_dir / "height_source_percentage_by_parcel.csv", index=False)

print(f"Aggregate outputs saved to: {aggregate_dir.resolve()}")
